[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/trim-filter-messages.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239435-lesson-4-trim-and-filter-messages)

# 过滤和修剪消息

## 回顾

现在，我们对几个方面有了更深入的理解：

* 如何自定义图状态模式
* 如何定义自定义状态reducer
* 如何使用多重图状态模式

## 目标

现在，我们可以开始在LangGraph中将这些概念与模型一起使用！
 
在接下来的几节中，我们将构建一个具有长期记忆的聊天机器人。

因为我们的聊天机器人将使用消息，让我们首先更多地讨论在图状态中处理消息的高级方法。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph langchain_openai

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

我们将使用[LangSmith](https://docs.smith.langchain.com/)进行[追踪](https://docs.smith.langchain.com/concepts/tracing)。

我们将记录到一个项目`langchain-academy`。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

## 消息作为状态

首先，让我们定义一些消息。

In [ ]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage
messages = [AIMessage(f"所以你说你在研究海洋哺乳动物？", name="Bot")]
messages.append(HumanMessage(f"是的，我了解鲸鱼。但是我还应该了解其他什么？", name="Lance"))

for m in messages:
    m.pretty_print()

回想一下，我们可以将它们传递给聊天模型。

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")
llm.invoke(messages)

我们可以在一个使用`MessagesState`的简单图中运行我们的聊天模型。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph, START, END

# 节点
def chat_model_node(state: MessagesState):
    return {"messages": llm.invoke(state["messages"])}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("chat_model", chat_model_node)
builder.add_edge(START, "chat_model")
builder.add_edge("chat_model", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
output = graph.invoke({'messages': messages})
for m in output['messages']:
    m.pretty_print()

## Reducer

处理消息时的一个实际挑战是管理长时间运行的对话。

如果我们不小心，长时间运行的对话会导致高token使用量和延迟，因为我们将不断增长的消息列表传递给模型。

我们有几种方法来解决这个问题。

首先，回想我们看到的使用`RemoveMessage`和`add_messages` reducer的技巧。

In [ ]:
from langchain_core.messages import RemoveMessage

# 节点
def filter_messages(state: MessagesState):
    # 删除除最近2条消息之外的所有消息
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"messages": delete_messages}

def chat_model_node(state: MessagesState):    
    return {"messages": [llm.invoke(state["messages"])]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("filter", filter_messages)
builder.add_node("chat_model", chat_model_node)
builder.add_edge(START, "filter")
builder.add_edge("filter", "chat_model")
builder.add_edge("chat_model", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# 带有序言的消息列表
messages = [AIMessage("嗨。", name="Bot", id="1")]
messages.append(HumanMessage("嗨。", name="Lance", id="2"))
messages.append(AIMessage("所以你说你在研究海洋哺乳动物？", name="Bot", id="3"))
messages.append(HumanMessage("是的，我了解鲸鱼。但是我还应该了解其他什么？", name="Lance", id="4"))

# 调用
output = graph.invoke({'messages': messages})
for m in output['messages']:
    m.pretty_print()

## 过滤消息

如果您不需要或不想修改图状态，您可以只过滤传递给聊天模型的消息。

例如，只传递一个过滤的列表：`llm.invoke(messages[-1:])`给模型。

In [ ]:
# 节点
def chat_model_node(state: MessagesState):
    return {"messages": [llm.invoke(state["messages"][-1:])]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("chat_model", chat_model_node)
builder.add_edge(START, "chat_model")
builder.add_edge("chat_model", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

让我们取我们现有的消息列表，附加上面的LLM响应，并附加一个后续问题。

In [ ]:
messages.append(output['messages'][-1])
messages.append(HumanMessage(f"告诉我更多关于独角鲸的信息！", name="Lance"))

In [ ]:
for m in messages:
    m.pretty_print()

In [ ]:
# 调用，使用消息过滤
output = graph.invoke({'messages': messages})
for m in output['messages']:
    m.pretty_print()

状态有所有的消息。

但是，让我们查看LangSmith追踪以查看模型调用只使用最后一条消息：

https://smith.langchain.com/public/75aca3ce-ef19-4b92-94be-0178c7a660d9/r

## 修剪消息

另一种方法是基于设定的token数量[修剪消息](https://python.langchain.com/v0.2/docs/how_to/trim_messages/#getting-the-last-max_tokens-tokens)。

这将消息历史限制为指定数量的token。

虽然过滤只返回代理之间消息的事后子集，修剪限制聊天模型可以用来响应的token数量。

请参见下面的`trim_messages`。

In [ ]:
from langchain_core.messages import trim_messages

# 节点
def chat_model_node(state: MessagesState):
    messages = trim_messages(
            state["messages"],
            max_tokens=100,
            strategy="last",
            token_counter=ChatOpenAI(model="gpt-4o"),
            allow_partial=False,
        )
    return {"messages": [llm.invoke(messages)]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("chat_model", chat_model_node)
builder.add_edge(START, "chat_model")
builder.add_edge("chat_model", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
messages.append(output['messages'][-1])
messages.append(HumanMessage(f"告诉我虎鲸住在哪里！", name="Lance"))

In [ ]:
# 修剪消息的示例
trim_messages(
            messages,
            max_tokens=100,
            strategy="last",
            token_counter=ChatOpenAI(model="gpt-4o"),
            allow_partial=False
        )

In [ ]:
# 调用，在chat_model_node中使用消息修剪
messages_out_trim = graph.invoke({'messages': messages})

让我们查看LangSmith追踪以查看模型调用：

https://smith.langchain.com/public/b153f7e9-f1a5-4d60-8074-f0d7ab5b42ef/r